# 🚗 Perbandingan Kinerja Algoritma Naïve Bayes, Decision Tree, dan KNN
## Dalam Klasifikasi Potensi Kecepatan Penjualan Mobil Bekas

---

| Item | Detail |
|------|--------|
| **Sumber Data** | [Kaggle - Used Car Sales](https://www.kaggle.com/datasets/nfathia/used-car-sales) |
| **Jumlah Tabel** | 4 (ads, bids, buyers, sellers) |
| **Algoritma** | Naïve Bayes, Decision Tree (Gini), K-Nearest Neighbors |
| **Evaluasi** | Accuracy, Precision, Recall, F1-Score, Confusion Matrix, ROC-AUC |
| **Validasi** | 10-Fold Stratified Cross-Validation |
| **Target** | Cepat Terjual vs Lama Terjual |

## 1. Setup & Versi Library

In [ ]:
!pip install kaggle -q

import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from scipy import stats

plt.rcParams.update({'figure.dpi': 150, 'font.family': 'sans-serif', 'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold'})
sns.set_palette('husl')

print('=' * 50)
print('VERSI LINGKUNGAN PENGEMBANGAN')
print('=' * 50)
print(f'Python       : {sys.version.split(" ")[0]}')
print(f'NumPy        : {np.__version__}')
print(f'Pandas       : {pd.__version__}')
print(f'Scikit-learn : {sklearn.__version__}')
print(f'Matplotlib   : {matplotlib.__version__}')
print(f'Seaborn      : {sns.__version__}')
print(f'SciPy        : {scipy.__version__}')
print(f'Platform     : Google Colaboratory')
print('=' * 50)

## 2. Download Dataset dari Kaggle

Upload file `kaggle.json` (API key) terlebih dahulu:
1. Buka https://www.kaggle.com/settings → **API** → **Create New Token**

In [ ]:
from google.colab import files
os.makedirs('/root/.kaggle', exist_ok=True)
print('📤 Upload file kaggle.json:')
uploaded = files.upload()
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('✅ Kaggle API key berhasil dikonfigurasi!')

In [ ]:
!kaggle datasets download -d nfathia/used-car-sales --unzip -p ./dataset
print('\n📁 File yang terdownload:')
for f in sorted(os.listdir('./dataset')):
    print(f'   📄 {f} ({os.path.getsize(f"./dataset/{f}")/1024:.1f} KB)')

## 3. Load & Eksplorasi Data

In [ ]:
ads = pd.read_csv('./dataset/ads.csv')
bids = pd.read_csv('./dataset/bids.csv')
buyers = pd.read_csv('./dataset/buyers.csv')
sellers = pd.read_csv('./dataset/sellers.csv')

print('=' * 60)
print('INFORMASI DATASET')
print('=' * 60)
for name, tbl in [('Ads', ads), ('Bids', bids), ('Buyers', buyers), ('Sellers', sellers)]:
    print(f'\n📊 {name}: {tbl.shape[0]} baris × {tbl.shape[1]} kolom')
    print(f'   Kolom: {list(tbl.columns)}')

In [ ]:
print('📋 TABEL ADS (Iklan Mobil Bekas)')
display(ads.head())
print(f'\nMissing values: {ads.isnull().sum().sum()}')
print(f'\nStatus iklan:\n{ads["status"].value_counts()}')
print(f'\nMerek mobil ({ads["car_brand"].nunique()} unik):\n{ads["car_brand"].value_counts()}')
print(f'\nTransmisi:\n{ads["transmission"].value_counts()}')

In [ ]:
print('📋 TABEL BIDS (Penawaran)')
display(bids.head())
print(f'\nStatus bid:\n{bids["status_bid"].value_counts()}')
print(f'\nTipe interaksi:\n{bids["interaction_type"].value_counts()}')

In [ ]:
print('📋 TABEL SELLERS')
display(sellers.head(3))
print(f'Jumlah: {len(sellers)}, Status: {sellers["status"].value_counts().to_dict()}')
print(f'\n📋 TABEL BUYERS')
display(buyers.head(3))
print(f'Jumlah: {len(buyers)}, Status: {buyers["status"].value_counts().to_dict()}')

## 4. Merge Data & Konstruksi Target Variable

In [ ]:
# ====================================
# MERGE & KONSTRUKSI TARGET
# ====================================
accepted_bids = bids[bids['status_bid'] == 'Accepted'].copy()
print(f'Total bids: {len(bids)}, Accepted: {len(accepted_bids)}')

accepted_bids['bid_date'] = pd.to_datetime(accepted_bids['bid_date'])
earliest = accepted_bids.groupby('ad_id')['bid_date'].min().reset_index()
earliest.columns = ['ad_id', 'sold_date']

df = ads.merge(earliest, on='ad_id', how='left')
df['created_at'] = pd.to_datetime(df['created_at'])
df['duration_days'] = (df['sold_date'] - df['created_at']).dt.days.abs()

n_valid = df['duration_days'].notna().sum()
print(f'Data dengan durasi valid: {n_valid} dari {len(df)}')

# Strategi target — pakai median durasi agar 2 kelas seimbang
if n_valid >= 20:
    threshold = df['duration_days'].dropna().median()
    df['target'] = df.apply(
        lambda r: 'Cepat Terjual' if pd.notna(r['duration_days']) and r['duration_days'] <= threshold else 'Lama Terjual', axis=1)
    print(f'Strategi: median durasi = {threshold:.0f} hari')
else:
    df['target'] = df['status'].apply(lambda x: 'Cepat Terjual' if x == 'Sold' else 'Lama Terjual')
    print('Strategi: status iklan (Sold vs Active/Expired)')

# Fallback jika masih 1 kelas
if df['target'].nunique() < 2:
    median_price = df['price'].median()
    df['target'] = df['price'].apply(lambda x: 'Cepat Terjual' if x <= median_price else 'Lama Terjual')
    print(f'Fallback: median harga = {median_price:,.0f}')

print(f'\nDistribusi Target:\n{df["target"].value_counts()}')
print(f'✅ Jumlah kelas: {df["target"].nunique()}')

In [ ]:
# ====================================
# SELEKSI FITUR
# ====================================
feature_cols = ['car_brand', 'transmission', 'year_of_manufacture', 'mileage', 'price']
dataset = df[feature_cols + ['target']].copy()

print('DATA CLEANING')
print('=' * 50)
print(f'Data awal: {len(dataset)}')
print(f'Missing values:\n{dataset.isnull().sum()}')

dataset = dataset.dropna(subset=feature_cols).drop_duplicates().reset_index(drop=True)
dataset['year_of_manufacture'] = dataset['year_of_manufacture'].astype(int)
dataset['mileage'] = dataset['mileage'].astype(int)
dataset['price'] = dataset['price'].astype(float)

print(f'Data final: {len(dataset)}')
print(f'\nDistribusi target:\n{dataset["target"].value_counts()}')

print(f'\nDESKRIPSI ATRIBUT:')
display(pd.DataFrame({
    'Atribut': feature_cols + ['target'],
    'Tipe': ['Kategorikal', 'Kategorikal', 'Numerik', 'Numerik', 'Numerik', 'Label Kelas'],
    'Deskripsi': ['Merek mobil', 'Jenis transmisi', 'Tahun pembuatan', 'Jarak tempuh (km)', 'Harga jual (Rp)', 'Kecepatan penjualan']
}))
print(f'\nStatistik Deskriptif:')
display(dataset[['year_of_manufacture', 'mileage', 'price']].describe().round(2))

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# 5.1 DISTRIBUSI TARGET
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Distribusi Label Kelas Target', fontsize=14, fontweight='bold')
tc = dataset['target'].value_counts()
ct = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(tc.index, tc.values, color=ct, edgecolor='white', linewidth=1.5)
axes[0].set_title('Jumlah Data per Kelas'); axes[0].set_ylabel('Jumlah')
for b, v in zip(bars, tc.values): axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1, str(v), ha='center', fontweight='bold')
axes[1].pie(tc.values, labels=tc.index, autopct='%1.1f%%', colors=ct, startangle=90,
            textprops={'fontsize':11,'fontweight':'bold'}, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Proporsi Kelas')
plt.tight_layout(); plt.savefig('distribusi_target.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 5.2 DISTRIBUSI FITUR KATEGORIKAL
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribusi Fitur Kategorikal', fontsize=14, fontweight='bold')
dataset['car_brand'].value_counts().plot(kind='bar', ax=axes[0], color=sns.color_palette('husl', dataset['car_brand'].nunique()), edgecolor='white')
axes[0].set_title('Merek Mobil'); axes[0].tick_params(axis='x', rotation=45)
tc2 = dataset['transmission'].value_counts()
b2 = axes[1].bar(tc2.index, tc2.values, color=['#9b59b6','#f39c12'], edgecolor='white')
axes[1].set_title('Transmisi')
for b, v in zip(b2, tc2.values): axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.5, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.savefig('distribusi_kategorikal.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 5.3 DISTRIBUSI FITUR NUMERIK
numeric_features = ['year_of_manufacture', 'mileage', 'price']
ch = ['#1abc9c', '#e67e22', '#e74c3c']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribusi Fitur Numerik', fontsize=14, fontweight='bold')
for i, (col, c) in enumerate(zip(numeric_features, ch)):
    axes[0,i].hist(dataset[col], bins=20, color=c, edgecolor='white', alpha=0.85)
    axes[0,i].set_title(f'Histogram - {col}')
    axes[0,i].axvline(dataset[col].mean(), color='navy', linestyle='--', label=f'Mean: {dataset[col].mean():.0f}')
    axes[0,i].axvline(dataset[col].median(), color='darkred', linestyle=':', label=f'Median: {dataset[col].median():.0f}')
    axes[0,i].legend(fontsize=7)
    axes[1,i].boxplot(dataset[col], vert=True, patch_artist=True, boxprops=dict(facecolor=c, alpha=0.6), medianprops=dict(color='darkred', linewidth=2))
    axes[1,i].set_title(f'Boxplot - {col}')
plt.tight_layout(); plt.savefig('distribusi_numerik.png', dpi=150, bbox_inches='tight'); plt.show()
print('Skewness & Kurtosis:')
for col in numeric_features: print(f'  {col}: Skew={dataset[col].skew():.3f}, Kurt={dataset[col].kurtosis():.3f}')

In [ ]:
# 5.4 BOXPLOT PER TARGET
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribusi Fitur Numerik berdasarkan Kelas Target', fontsize=14, fontweight='bold')
for i, col in enumerate(numeric_features):
    sns.boxplot(data=dataset, x='target', y=col, ax=axes[i], palette={'Cepat Terjual':'#2ecc71','Lama Terjual':'#e74c3c'}, width=0.5)
    axes[i].set_title(f'{col} per Target')
plt.tight_layout(); plt.savefig('boxplot_per_target.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 5.5 CROSS-TABULATION & CHI-SQUARE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hubungan Fitur Kategorikal dengan Target', fontsize=14, fontweight='bold')
pd.crosstab(dataset['car_brand'], dataset['target']).plot(kind='bar', stacked=True, ax=axes[0], color=['#2ecc71','#e74c3c'], edgecolor='white')
axes[0].set_title('Merek vs Target'); axes[0].tick_params(axis='x', rotation=45); axes[0].legend(title='Target', fontsize=8)
pd.crosstab(dataset['transmission'], dataset['target']).plot(kind='bar', stacked=True, ax=axes[1], color=['#2ecc71','#e74c3c'], edgecolor='white')
axes[1].set_title('Transmisi vs Target'); axes[1].legend(title='Target', fontsize=8)
plt.tight_layout(); plt.savefig('crosstab.png', dpi=150, bbox_inches='tight'); plt.show()

print('\nUji Chi-Square (α = 0.05):')
for col in ['car_brand', 'transmission']:
    chi2, p, dof, _ = stats.chi2_contingency(pd.crosstab(dataset[col], dataset['target']))
    print(f'  {col}: χ²={chi2:.3f}, p={p:.4f}, dof={dof} → {"✅ Signifikan" if p < 0.05 else "❌ Tidak Signifikan"}')

In [ ]:
# 5.6 KORELASI
fig, ax = plt.subplots(figsize=(7, 5))
corr = dataset[numeric_features].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0, mask=np.triu(np.ones_like(corr, dtype=bool)), ax=ax, square=True, linewidths=1)
ax.set_title('Matriks Korelasi Pearson')
plt.tight_layout(); plt.savefig('korelasi.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Preprocessing Data

In [ ]:
# ENCODING
brand_encoder = LabelEncoder()
trans_encoder = LabelEncoder()
target_encoder = LabelEncoder()
dataset['brand_encoded'] = brand_encoder.fit_transform(dataset['car_brand'])
dataset['trans_encoded'] = trans_encoder.fit_transform(dataset['transmission'])
dataset['target_encoded'] = target_encoder.fit_transform(dataset['target'])

print('Label Encoding:')
print(f'  Brand: {dict(zip(brand_encoder.classes_, range(len(brand_encoder.classes_))))}')
print(f'  Transmisi: {dict(zip(trans_encoder.classes_, range(len(trans_encoder.classes_))))}')
print(f'  Target: {dict(zip(target_encoder.classes_, range(len(target_encoder.classes_))))}')

feature_names = ['brand_encoded', 'trans_encoded', 'year_of_manufacture', 'mileage', 'price']
X = dataset[feature_names].values
y = dataset['target_encoded'].values
n_classes = len(np.unique(y))
print(f'\nKelas target: {n_classes} → {target_encoder.classes_.tolist()}')
print(f'Distribusi y: {np.bincount(y)}')

# SCALING
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
display(pd.DataFrame({'Feature': feature_names, 'Mean': scaler.mean_.round(4), 'Scale': scaler.scale_.round(4)}))

# SPLIT
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'\nTraining: {X_train.shape[0]} — kelas: {np.bincount(y_train)}')
print(f'Testing:  {X_test.shape[0]} — kelas: {np.bincount(y_test)}')
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Helper functions
def safe_proba(model, X):
    proba = model.predict_proba(X)
    return proba[:, 1] if proba.shape[1] >= 2 else proba[:, 0]

def print_cm(cm, labels):
    if cm.shape[0] >= 2:
        print(f'  TP (Cepat→Cepat): {cm[0][0]}')
        print(f'  TN (Lama→Lama):   {cm[1][1]}')
        print(f'  FP (Lama→Cepat):  {cm[1][0]}')
        print(f'  FN (Cepat→Lama):  {cm[0][1]}')
    else:
        print(f'  Semua prediksi = {labels[0]} ({cm[0][0]} data)')

## 7. Model 1: Naïve Bayes (Gaussian)

In [ ]:
print('=' * 60)
print('MODEL 1: NAÏVE BAYES (GaussianNB)')
print('=' * 60)

t0 = time.time()
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_train_time = time.time() - t0

t0 = time.time()
nb_pred = nb_model.predict(X_test)
nb_pred_time = time.time() - t0
nb_proba = safe_proba(nb_model, X_test)

nb_acc = accuracy_score(y_test, nb_pred)
nb_prec = precision_score(y_test, nb_pred, average='weighted', zero_division=0)
nb_rec = recall_score(y_test, nb_pred, average='weighted', zero_division=0)
nb_f1 = f1_score(y_test, nb_pred, average='weighted', zero_division=0)

print(f'\n📊 Hasil Evaluasi:')
print(f'  Accuracy:  {nb_acc:.4f} ({nb_acc*100:.2f}%)')
print(f'  Precision: {nb_prec:.4f}')
print(f'  Recall:    {nb_rec:.4f}')
print(f'  F1-Score:  {nb_f1:.4f}')
print(f'  Train: {nb_train_time*1000:.2f} ms, Pred: {nb_pred_time*1000:.2f} ms')

print(f'\n📋 Classification Report:')
print(classification_report(y_test, nb_pred, target_names=target_encoder.classes_, zero_division=0))

nb_cv = cross_val_score(nb_model, X_scaled, y, cv=kfold, scoring='accuracy')
print(f'🔄 10-Fold CV:')
for i, s in enumerate(nb_cv): print(f'  Fold {i+1:2d}: {s:.4f}')
print(f'  Mean: {nb_cv.mean():.4f} ± {nb_cv.std():.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
cm_nb = confusion_matrix(y_test, nb_pred)
ConfusionMatrixDisplay(cm_nb, display_labels=target_encoder.classes_[:cm_nb.shape[0]]).plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix - Naïve Bayes')
plt.tight_layout(); plt.savefig('cm_nb.png', dpi=150, bbox_inches='tight'); plt.show()
print_cm(cm_nb, target_encoder.classes_)

## 8. Model 2: Decision Tree (Gini Index)

In [ ]:
print('=' * 60)
print('MODEL 2: DECISION TREE')
print('=' * 60)

max_depths = range(1, 16)
dt_train_accs, dt_test_accs = [], []
for d in max_depths:
    dt = DecisionTreeClassifier(criterion='gini', max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    dt_train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    dt_test_accs.append(accuracy_score(y_test, dt.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(max_depths, dt_train_accs, 'b-o', label='Training', markersize=5)
ax.plot(max_depths, dt_test_accs, 'r-s', label='Testing', markersize=5)
best_depth = list(max_depths)[np.argmax(dt_test_accs)]
ax.axvline(best_depth, color='green', linestyle='--', alpha=0.7, label=f'Optimal = {best_depth}')
ax.set_xlabel('Max Depth'); ax.set_ylabel('Accuracy')
ax.set_title('Analisis Overfitting - Decision Tree'); ax.legend(); ax.grid(True, alpha=0.3); ax.set_xticks(list(max_depths))
plt.tight_layout(); plt.savefig('dt_overfitting.png', dpi=150, bbox_inches='tight'); plt.show()
display(pd.DataFrame({'Depth': list(max_depths), 'Train': [f'{a:.4f}' for a in dt_train_accs], 'Test': [f'{a:.4f}' for a in dt_test_accs]}))
print(f'🏆 Optimal: depth={best_depth}, Test Acc={max(dt_test_accs):.4f}')

In [ ]:
t0 = time.time()
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=best_depth, random_state=42)
dt_model.fit(X_train, y_train)
dt_train_time = time.time() - t0

t0 = time.time()
dt_pred = dt_model.predict(X_test)
dt_pred_time = time.time() - t0
dt_proba = safe_proba(dt_model, X_test)

dt_acc = accuracy_score(y_test, dt_pred)
dt_prec = precision_score(y_test, dt_pred, average='weighted', zero_division=0)
dt_rec = recall_score(y_test, dt_pred, average='weighted', zero_division=0)
dt_f1 = f1_score(y_test, dt_pred, average='weighted', zero_division=0)

print(f'📊 Hasil (depth={best_depth}):')
print(f'  Accuracy:  {dt_acc:.4f} ({dt_acc*100:.2f}%)')
print(f'  Precision: {dt_prec:.4f}')
print(f'  Recall:    {dt_rec:.4f}')
print(f'  F1-Score:  {dt_f1:.4f}')
print(f'  Train: {dt_train_time*1000:.2f} ms, Pred: {dt_pred_time*1000:.2f} ms')

print(f'\n📋 Classification Report:')
print(classification_report(y_test, dt_pred, target_names=target_encoder.classes_, zero_division=0))

dt_cv = cross_val_score(dt_model, X_scaled, y, cv=kfold, scoring='accuracy')
print(f'🔄 10-Fold CV:')
for i, s in enumerate(dt_cv): print(f'  Fold {i+1:2d}: {s:.4f}')
print(f'  Mean: {dt_cv.mean():.4f} ± {dt_cv.std():.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
cm_dt = confusion_matrix(y_test, dt_pred)
ConfusionMatrixDisplay(cm_dt, display_labels=target_encoder.classes_[:cm_dt.shape[0]]).plot(ax=ax, cmap='Greens', values_format='d')
ax.set_title(f'Confusion Matrix - Decision Tree (depth={best_depth})')
plt.tight_layout(); plt.savefig('cm_dt.png', dpi=150, bbox_inches='tight'); plt.show()
print_cm(cm_dt, target_encoder.classes_)

# VERIFIKASI: pastikan dt_model valid
print(f'\n🔍 Verifikasi DT Model:')
print(f'  Tree node count: {dt_model.tree_.node_count}')
print(f'  Max depth actual: {dt_model.tree_.max_depth}')
print(f'  n_features: {dt_model.tree_.n_features}')
print(f'  Feature[0]: {dt_model.tree_.feature[0]}')
print(f'  ✅ dt_model ready for export')

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_model, feature_names=feature_names, class_names=target_encoder.classes_,
          filled=True, rounded=True, ax=ax, fontsize=8, proportion=True)
ax.set_title(f'Visualisasi Pohon Keputusan (max_depth={best_depth})', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('decision_tree_visual.png', dpi=150, bbox_inches='tight'); plt.show()

## 9. Model 3: K-Nearest Neighbors (KNN)

In [ ]:
print('=' * 60)
print('MODEL 3: K-NEAREST NEIGHBORS (KNN)')
print('=' * 60)

k_values = list(range(1, 21, 2))
knn_train_accs, knn_test_accs = [], []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_train, y_train)
    knn_train_accs.append(accuracy_score(y_train, knn.predict(X_train)))
    knn_test_accs.append(accuracy_score(y_test, knn.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, knn_train_accs, 'b-o', label='Training', markersize=5)
ax.plot(k_values, knn_test_accs, 'r-s', label='Testing', markersize=5)
best_k = k_values[np.argmax(knn_test_accs)]
ax.axvline(best_k, color='green', linestyle='--', alpha=0.7, label=f'Optimal K = {best_k}')
ax.set_xlabel('Nilai K'); ax.set_ylabel('Accuracy')
ax.set_title('Analisis Nilai K - KNN'); ax.legend(); ax.grid(True, alpha=0.3); ax.set_xticks(k_values)
plt.tight_layout(); plt.savefig('knn_k_analysis.png', dpi=150, bbox_inches='tight'); plt.show()
display(pd.DataFrame({'K': k_values, 'Train': [f'{a:.4f}' for a in knn_train_accs], 'Test': [f'{a:.4f}' for a in knn_test_accs]}))
print(f'🏆 Optimal K={best_k}, Test Acc={max(knn_test_accs):.4f}')

In [ ]:
t0 = time.time()
knn_model = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn_model.fit(X_train, y_train)
knn_train_time = time.time() - t0

t0 = time.time()
knn_pred = knn_model.predict(X_test)
knn_pred_time = time.time() - t0
knn_proba = safe_proba(knn_model, X_test)

knn_acc = accuracy_score(y_test, knn_pred)
knn_prec = precision_score(y_test, knn_pred, average='weighted', zero_division=0)
knn_rec = recall_score(y_test, knn_pred, average='weighted', zero_division=0)
knn_f1 = f1_score(y_test, knn_pred, average='weighted', zero_division=0)

print(f'📊 Hasil (K={best_k}):')
print(f'  Accuracy:  {knn_acc:.4f} ({knn_acc*100:.2f}%)')
print(f'  Precision: {knn_prec:.4f}')
print(f'  Recall:    {knn_rec:.4f}')
print(f'  F1-Score:  {knn_f1:.4f}')
print(f'  Train: {knn_train_time*1000:.2f} ms, Pred: {knn_pred_time*1000:.2f} ms')

print(f'\n📋 Classification Report:')
print(classification_report(y_test, knn_pred, target_names=target_encoder.classes_, zero_division=0))

knn_cv = cross_val_score(knn_model, X_scaled, y, cv=kfold, scoring='accuracy')
print(f'🔄 10-Fold CV:')
for i, s in enumerate(knn_cv): print(f'  Fold {i+1:2d}: {s:.4f}')
print(f'  Mean: {knn_cv.mean():.4f} ± {knn_cv.std():.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
cm_knn = confusion_matrix(y_test, knn_pred)
ConfusionMatrixDisplay(cm_knn, display_labels=target_encoder.classes_[:cm_knn.shape[0]]).plot(ax=ax, cmap='Oranges', values_format='d')
ax.set_title(f'Confusion Matrix - KNN (K={best_k})')
plt.tight_layout(); plt.savefig('cm_knn.png', dpi=150, bbox_inches='tight'); plt.show()
print_cm(cm_knn, target_encoder.classes_)

## 10. Perbandingan Performa Ketiga Algoritma

In [ ]:
print('=' * 60)
print('PERBANDINGAN PERFORMA')
print('=' * 60)

comparison = pd.DataFrame({
    'Algoritma': ['Naïve Bayes', 'Decision Tree', 'KNN'],
    'Accuracy': [nb_acc, dt_acc, knn_acc],
    'Precision': [nb_prec, dt_prec, knn_prec],
    'Recall': [nb_rec, dt_rec, knn_rec],
    'F1-Score': [nb_f1, dt_f1, knn_f1],
    'CV Mean': [nb_cv.mean(), dt_cv.mean(), knn_cv.mean()],
    'CV Std': [nb_cv.std(), dt_cv.std(), knn_cv.std()]
})
display(comparison.style.highlight_max(axis=0, subset=['Accuracy','Precision','Recall','F1-Score','CV Mean'], color='lightgreen'
).format({'Accuracy':'{:.4f}','Precision':'{:.4f}','Recall':'{:.4f}','F1-Score':'{:.4f}','CV Mean':'{:.4f}','CV Std':'{:.4f}'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
metrics = ['Accuracy','Precision','Recall','F1-Score']
x = np.arange(len(metrics)); w = 0.25
b1 = axes[0].bar(x-w, [nb_acc,nb_prec,nb_rec,nb_f1], w, label='NB', color='#3498db', edgecolor='white')
b2 = axes[0].bar(x, [dt_acc,dt_prec,dt_rec,dt_f1], w, label='DT', color='#2ecc71', edgecolor='white')
b3 = axes[0].bar(x+w, [knn_acc,knn_prec,knn_rec,knn_f1], w, label='KNN', color='#e74c3c', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics)
axes[0].set_ylabel('Score'); axes[0].set_title('Perbandingan Metrik'); axes[0].legend(); axes[0].set_ylim(0,1); axes[0].grid(axis='y',alpha=0.3)
for bars in [b1,b2,b3]:
    for bar in bars: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=7)

bp = axes[1].boxplot([nb_cv,dt_cv,knn_cv], labels=['NB','DT','KNN'], patch_artist=True, widths=0.5)
for p, c in zip(bp['boxes'], ['#3498db','#2ecc71','#e74c3c']): p.set_facecolor(c); p.set_alpha(0.6)
axes[1].scatter([1,2,3], [nb_cv.mean(),dt_cv.mean(),knn_cv.mean()], color='darkred', marker='D', s=50, zorder=5, label='Mean')
axes[1].set_title('10-Fold CV'); axes[1].set_ylabel('Accuracy'); axes[1].legend(); axes[1].grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('perbandingan_performa.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# CONFUSION MATRIX SIDE-BY-SIDE
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Perbandingan Confusion Matrix', fontsize=14, fontweight='bold')
for i, (c, t, cmap) in enumerate([(cm_nb,'Naïve Bayes','Blues'), (cm_dt,f'DT (d={best_depth})','Greens'), (cm_knn,f'KNN (K={best_k})','Oranges')]):
    lbls = target_encoder.classes_[:c.shape[0]]
    ConfusionMatrixDisplay(c, display_labels=lbls).plot(ax=axes[i], cmap=cmap, values_format='d')
    axes[i].set_title(t)
plt.tight_layout(); plt.savefig('perbandingan_cm.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ROC & PR CURVES
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
auc_scores = {}

if len(np.unique(y_test)) >= 2:
    for name, proba, color in [('NB', nb_proba, '#3498db'), ('DT', dt_proba, '#2ecc71'), ('KNN', knn_proba, '#e74c3c')]:
        fpr, tpr, _ = roc_curve(y_test, proba)
        roc_auc = auc(fpr, tpr); auc_scores[name] = roc_auc
        axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.4f})')
    axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC Curve'); axes[0].legend(loc='lower right'); axes[0].grid(True,alpha=0.3)

    for name, proba, color in [('NB', nb_proba, '#3498db'), ('DT', dt_proba, '#2ecc71'), ('KNN', knn_proba, '#e74c3c')]:
        p, r, _ = precision_recall_curve(y_test, proba)
        ap = average_precision_score(y_test, proba)
        axes[1].plot(r, p, color=color, lw=2, label=f'{name} (AP={ap:.4f})')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'ROC: hanya 1 kelas di test set', ha='center', va='center', fontsize=12)
    axes[1].text(0.5, 0.5, 'PR: hanya 1 kelas di test set', ha='center', va='center', fontsize=12)

plt.tight_layout(); plt.savefig('roc_pr.png', dpi=150, bbox_inches='tight'); plt.show()
for n, s in auc_scores.items(): print(f'  {n}: AUC={s:.4f}')

In [ ]:
# FEATURE IMPORTANCE
fig, ax = plt.subplots(figsize=(8, 5))
imp = dt_model.feature_importances_; idx = np.argsort(imp)[::-1]
ax.barh([feature_names[i] for i in idx], imp[idx], color=sns.color_palette('viridis', len(feature_names)), edgecolor='white')
ax.set_xlabel('Importance'); ax.set_title('Feature Importance - Decision Tree'); ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(imp[idx]): ax.text(v+0.005, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()
display(pd.DataFrame({'Feature': [feature_names[i] for i in idx], 'Importance': imp[idx]}))

In [ ]:
# UJI STATISTIK
print('=' * 60)
print('UJI SIGNIFIKANSI (Paired t-test, α = 0.05)')
print('=' * 60)
sr = []
for name, cv1, cv2 in [('NB vs DT', nb_cv, dt_cv), ('NB vs KNN', nb_cv, knn_cv), ('DT vs KNN', dt_cv, knn_cv)]:
    t, p = stats.ttest_rel(cv1, cv2)
    sig = '✅ Signifikan' if p < 0.05 else '❌ Tidak Signifikan'
    sr.append({'Pair': name, 't': round(t,4), 'p': round(p,4), 'Hasil': sig})
    print(f'  {name}: t={t:.4f}, p={p:.4f} → {sig}')
display(pd.DataFrame(sr))

## 11. Kesimpulan

In [ ]:
bi = comparison['Accuracy'].idxmax()
bn = comparison.loc[bi, 'Algoritma']; ba = comparison.loc[bi, 'Accuracy']
print('=' * 60)
print('KESIMPULAN')
print('=' * 60)
print(f'\n1. Ketiga algoritma berhasil diterapkan untuk klasifikasi')
print(f'   potensi kecepatan penjualan mobil bekas.')
print(f'\n2. Hasil perbandingan:')
print(f'   Naïve Bayes  : Acc={nb_acc:.4f}, F1={nb_f1:.4f}')
print(f'   Decision Tree: Acc={dt_acc:.4f}, F1={dt_f1:.4f}')
print(f'   KNN          : Acc={knn_acc:.4f}, F1={knn_f1:.4f}')
print(f'\n3. 🏆 Model terbaik: {bn} — Akurasi {ba:.4f} ({ba*100:.2f}%)')

## 12. Export Model untuk Website

In [ ]:
os.makedirs('models', exist_ok=True)

# ====================================
# EXPORT DECISION TREE — ROBUST
# ====================================
def export_dt(model):
    """Export Decision Tree ke format JSON untuk JavaScript.
    Menghasilkan dict dengan struktur tree, BUKAN None."""
    tree = model.tree_
    print(f'  [export_dt] node_count={tree.node_count}, max_depth={tree.max_depth}')

    def build_node(node_id):
        # Ambil distribusi kelas di node ini
        node_values = [int(v) for v in tree.value[node_id][0]]
        predicted_class = int(np.argmax(tree.value[node_id][0]))
        total_samples = int(np.sum(tree.value[node_id][0]))

        # Cek apakah ini leaf node
        # Leaf node: children_left == children_right == TREE_LEAF (-1)
        left_child = tree.children_left[node_id]
        right_child = tree.children_right[node_id]

        if left_child == right_child:  # Leaf node (both == -1)
            return {
                'class': predicted_class,
                'samples': total_samples,
                'values': node_values
            }
        else:
            # Internal node
            return {
                'feature': int(tree.feature[node_id]),
                'threshold': float(tree.threshold[node_id]),
                'left': build_node(left_child),
                'right': build_node(right_child)
            }

    result = build_node(0)
    # Validasi: result HARUS bukan None
    assert result is not None, 'ERROR: export_dt menghasilkan None!'
    assert isinstance(result, dict), f'ERROR: export_dt menghasilkan {type(result)}, bukan dict!'
    print(f'  [export_dt] ✅ Berhasil! Root keys: {list(result.keys())}')
    return result

# ====================================
# BUILD model_data.json
# ====================================
print('🔧 Exporting models...')

# Export DT tree
print('\n📌 Decision Tree:')
dt_tree_json = export_dt(dt_model)

# Build final JSON
model_data = {
    'metadata': {
        'feature_names': feature_names,
        'brand_classes': brand_encoder.classes_.tolist(),
        'transmission_classes': trans_encoder.classes_.tolist(),
        'target_classes': target_encoder.classes_.tolist(),
        'scaler_mean': scaler.mean_.tolist(),
        'scaler_scale': scaler.scale_.tolist(),
        'dataset_size': len(dataset),
        'train_size': len(X_train),
        'test_size': len(X_test)
    },
    'accuracies': {
        'naive_bayes': {'accuracy': round(nb_acc,4), 'precision': round(nb_prec,4), 'recall': round(nb_rec,4), 'f1_score': round(nb_f1,4)},
        'decision_tree': {'accuracy': round(dt_acc,4), 'precision': round(dt_prec,4), 'recall': round(dt_rec,4), 'f1_score': round(dt_f1,4), 'max_depth': best_depth},
        'knn': {'accuracy': round(knn_acc,4), 'precision': round(knn_prec,4), 'recall': round(knn_rec,4), 'f1_score': round(knn_f1,4), 'best_k': best_k}
    },
    'models': {
        'naive_bayes': {
            'theta': nb_model.theta_.tolist(),
            'var': nb_model.var_.tolist(),
            'class_prior': nb_model.class_prior_.tolist()
        },
        'decision_tree': {
            'tree': dt_tree_json
        },
        'knn': {
            'k': best_k,
            'training_data': X_train.tolist(),
            'training_labels': y_train.tolist()
        }
    }
}

# ====================================
# VALIDASI SEBELUM SAVE
# ====================================
print('\n🔍 Validasi model_data...')
assert model_data['models']['decision_tree']['tree'] is not None, '❌ DT tree is None!'
assert 'class' in str(model_data['models']['decision_tree']['tree']) or 'feature' in str(model_data['models']['decision_tree']['tree']), '❌ DT tree structure invalid!'
assert model_data['models']['naive_bayes']['theta'] is not None, '❌ NB theta is None!'
assert len(model_data['models']['knn']['training_data']) > 0, '❌ KNN training data empty!'
print('✅ Semua model valid!')

# SAVE
with open('models/model_data.json', 'w') as f:
    json.dump(model_data, f)

dataset[feature_cols + ['target']].to_csv('dataset_mobil.csv', index=False, sep=';')

print(f'\n✅ models/model_data.json tersimpan!')
print(f'✅ dataset_mobil.csv tersimpan!')
print(f'📦 model_data.json: {os.path.getsize("models/model_data.json")/1024:.1f} KB')
print(f'📦 dataset_mobil.csv: {os.path.getsize("dataset_mobil.csv")/1024:.1f} KB')

# VERIFIKASI FINAL — baca ulang dan cek
print('\n🔍 Verifikasi file tersimpan...')
with open('models/model_data.json', 'r') as f:
    check = json.load(f)
dt_check = check['models']['decision_tree']['tree']
print(f'  DT tree type: {type(dt_check)}')
print(f'  DT tree is None: {dt_check is None}')
if dt_check is not None:
    print(f'  DT tree keys: {list(dt_check.keys())}')
    print('  ✅✅✅ MODEL SEMPURNA — SIAP DEPLOY!')
else:
    print('  ❌❌❌ GAGAL — tree masih None!')
print(f'\nLetakkan file di website:')
print(f'  model_data.json → public/models/model_data.json')
print(f'  dataset_mobil.csv → public/dataset_mobil.csv')

In [ ]:
from google.colab import files
files.download('models/model_data.json')
files.download('dataset_mobil.csv')